Inicio de codigo: Importacion de librerias
Cosas a hacer:
- dataset
- eliminar nulos o imputar de alguna maneras, se puede descartar columnas y filas (a discutir si elimnar remotos)
- convertir todo a numerico
- normalizar datos (zscale)
- split de datos (entrenamiento y validacion)
 - Validacion: r2 y MSE
- modelo (min 4)
- dibujar distribucion


1. Importacion de librerias 

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Preprocesamiento y modelado
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# Algoritmos de regresión
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor

# Métricas
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Opcional: para ignorar warnings
import warnings
warnings.filterwarnings('ignore')


2. CARGA Y EXPLORACIÓN DEL DATASET

In [ ]:
df = pd.read_csv('job_salary_prediction_dataset.csv')
df.head()
print("Dimensiones:", df.shape)
print("\nTipos de datos:")
print(df.dtypes)
print("\nValores nulos:")
print(df.isnull().sum())
df = df.sample(frac=0.1, random_state=42)


3. PREPROCESAMIENTO Y DIVISIÓN DE DATOS

In [ ]:
# Caracteristicas separadas de el objetivo
X = df.drop('salary', axis=1)
y = df['salary']

# Dividir 80% para entrenamiento y el 20% restante para validacion
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Identificar que columas son numeros y cuales son texto
cat_cols = X.select_dtypes(include=['object']).columns
num_cols = X.select_dtypes(exclude=['object']).columns

# Transformador para obtener datos en forma de numeros y debidamente escalados
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
    ])

print(f"Muestras para entrenar: {X_train.shape[0]}")
print(f"Muestras para validar: {X_test.shape[0]}")


4. ENTRENAMIENTO Y EVALUACIÓN DE MODELOS

In [ ]:
modelos = {
    "1. Regresión Lineal": LinearRegression(),
    "2. K-Vecinos (KNN)": KNeighborsRegressor(n_neighbors=5),
    "3. Random Forest": RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1),
    "4. Gradient Boosting": GradientBoostingRegressor(n_estimators=50, random_state=42)
}

resultados = []

for nombre, algoritmo in modelos.items():
    print(f"Entrenando {nombre}...")
    
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', algoritmo)
    ])
    
    pipeline.fit(X_train, y_train)
    
    y_pred = pipeline.predict(X_test)
    
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    resultados.append({'Modelo': nombre, 'R2': r2, 'MSE': mse})
    print(f" -> R2 Score: {r2:.4f} | MSE: {mse:,.2f}\n")
    

5. OPTIMIZACIÓN DE HIPERPARAMETROS

In [ ]:
param_distributions = {
    'model__n_estimators': [20, 50, 100, 150],   # Numero de arboles
    'model__max_depth': [None, 10, 20, 30],      # Profundidad maxima
    'model__min_samples_split': [2, 5, 10]       # Muestras minimas para dividir un nodo
}

pipeline_rf = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(random_state=42, n_jobs=-1))
])

# Hacemos busqueda aleatoria con Cross-Validation (cv=3)
random_search = RandomizedSearchCV(
    estimator=pipeline_rf, 
    param_distributions=param_distributions, 
    n_iter=5,             # Probará 5 combinaciones al azar de la cuadrícula
    cv=3,                 # Validación cruzada de 3 pliegues (K-Fold)
    scoring='r2', 
    random_state=42,
    n_jobs=-1             # Usar todos los núcleos
)

print("Iniciando (Tuning) con Cross-Validation...")
random_search.fit(X_train, y_train)

# Resultados del Tuning
print(f"Mejores hiperparámetros encontrados:\n{random_search.best_params_}")
print(f"Mejor R2 con validación cruzada: {random_search.best_score_:.4f}")

# Guardamos el mejor modelo para usarlo después
mejor_rf = random_search.best_estimator_

6. INTEROPERABILIDAD (MEDIR QUE TAN IMPORTANTES SON CADA VARIABLE)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Sacar los nombres de las columnas
nombres_variables = mejor_rf.named_steps['preprocessor'].get_feature_names_out()

# Sacar los pesos de los árboles del Random Forest, basicamente la importancia de cada variable
importancias = mejor_rf.named_steps['model'].feature_importances_
importancias_absolutas = {col: 0.0 for col in X.columns}

for nombre_completo, importancia in zip(nombres_variables, importancias):
    for col_original in X.columns:
        if f"num__{col_original}" == nombre_completo or nombre_completo.startswith(f"cat__{col_original}_"):
            importancias_absolutas[col_original] += importancia
            break

# DF para ordenar los resultados
df_puras = pd.DataFrame(
    list(importancias_absolutas.items()), 
    columns=['Variable', 'Importancia']
).sort_values(by='Importancia', ascending=False)

# Indicar las 10 variables mas importantes
plt.figure(figsize=(10, 6))
plt.barh(df_puras['Variable'], df_puras['Importancia'], color='#2ca02c', edgecolor='black')
plt.gca().invert_yaxis() # Invertir para que la más importante quede arriba
plt.title('Importancia Global de las Variables Originales', fontsize=14, fontweight='bold')
plt.xlabel('Importancia Acumulada (Suma de sub-variables)', fontsize=12)
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

# Imprimir por pantalla el top 3 para análisis rápido
print("\nVariables mas importantes:")
for i, fila in df_puras.iterrows():
    print(f" - {fila['Variable']}: {fila['Importancia']*100:.2f}% de impacto total")